In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import os
import cv2
import numpy as np
import onnxruntime as ort
import rospy
from sensor_msgs.msg import Image
from cv_bridge import CvBridge, CvBridgeError

print(f"[+] ONNX Runtime Version: {ort.__version__}")

# Kiểm tra các Provider xử lý (Ưu tiên CUDA/TensorRT)
available_providers = ort.get_available_providers()
print(f"[+] Available Providers: {available_providers}")

if 'CUDAExecutionProvider' in available_providers:
    print("[✓] Đã sẵn sàng chạy ONNX trên GPU (CUDA)!")
else:
    print("[!] Cảnh báo: Không tìm thấy CUDA Provider, mô hình sẽ chạy bằng CPU.")

[+] ONNX Runtime Version: 1.10.0
[+] Available Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
[✓] Đã sẵn sàng chạy ONNX trên GPU (CUDA)!


In [4]:
# sim_path = os.path.abspath(os.path.join("..", "src", "simulation"))
# if sim_path not in sys.path:
#     sys.path.insert(0, sim_path)

In [5]:
class ONNXModelRunner:
    def __init__(self, onnx_path):
        print(f"[+] Đang nạp mô hình ONNX từ: {onnx_path}")
        
        # Thiết lập ưu tiên GPU
        providers = [
            ('CUDAExecutionProvider', {
                'device_id': 0,
            }),
            'CPUExecutionProvider'
        ]
        
        # Tải mô hình
        self.session = ort.InferenceSession(onnx_path, providers=providers)
        
        # Lấy thông tin Input/Output Tensors
        self.input_name = self.session.get_inputs()[0].name
        self.input_shape = self.session.get_inputs()[0].shape
        self.output_names = [out.name for out in self.session.get_outputs()]
        
        print(f"[✓] Nạp ONNX thành công!")
        print(f" -> Input Name: {self.input_name}")
        print(f" -> Input Shape mong đợi: {self.input_shape}")

    def preprocess(self, cv_img):
        # Lấy kích thước H, W từ shape (Thường là [1, 3, 640, 640] hoặc [1, 3, H, W])
        h = self.input_shape[2] if isinstance(self.input_shape[2], int) else 640
        w = self.input_shape[3] if isinstance(self.input_shape[3], int) else 640
        
        # Resize, BGR -> RGB, Normalize 0-1
        resized = cv2.resize(cv_img, (w, h))
        rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
        tensor = rgb.transpose((2, 0, 1)).astype(np.float32) / 255.0
        
        # Thêm Batch Dimension [1, 3, H, W]
        tensor = np.ascontiguousarray(np.expand_dims(tensor, axis=0))
        return tensor

    def infer(self, input_tensor):
        # Chạy inference
        outputs = self.session.run(self.output_names, {self.input_name: input_tensor})
        return outputs

print("[✓] Đã khởi tạo class ONNXModelRunner!")

[✓] Đã khởi tạo class ONNXModelRunner!


In [6]:
import cv2
import numpy as np

import rospy
from sensor_msgs.msg import LaserScan, Image
from jetracer.nvidia_racecar import NvidiaRacecar

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


In [7]:
car = NvidiaRacecar()

In [20]:
import rospy
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display
from sensor_msgs.msg import Image, CompressedImage
from typing import Any, Optional

# =========================================================
# 1. HÀM DECODE ÁNH CHUẨN (KHÔNG DÙNG CV_BRIDGE)
# =========================================================

def decode_image(image_msg: Any) -> Optional[np.ndarray]:
    """Decode a ROS Image/CompressedImage message into an OpenCV BGR frame."""
    try:
        # 1. Xử lý CompressedImage (JPEG / PNG)
        if hasattr(image_msg, 'format') or 'compressed' in getattr(image_msg, 'encoding', '').lower():
            np_arr = np.frombuffer(image_msg.data, np.uint8)
            frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
            return frame

        # 2. Xử lý Raw Image (sensor_msgs/Image)
        encoding = image_msg.encoding.lower()
        
        # Xử lý Depth / 16-bit
        if encoding in ['16uc1', 'mono16']:
            frame = np.frombuffer(image_msg.data, dtype=np.uint16)
            frame = frame.reshape(image_msg.height, image_msg.width)
            frame = cv2.normalize(frame, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
            frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
            return frame
        else:
            frame = np.frombuffer(image_msg.data, dtype=np.uint8)

        # Xử lý theo kênh màu
        if encoding in ['mono8', '8uc1']:
            frame = frame.reshape(image_msg.height, image_msg.width)
            frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
        elif encoding in ['bgr8', '8uc3']:
            frame = frame.reshape(image_msg.height, image_msg.width, 3)
        elif encoding in ['rgb8']:
            frame = frame.reshape(image_msg.height, image_msg.width, 3)
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        elif encoding in ['bgra8', '8uc4']:
            frame = frame.reshape(image_msg.height, image_msg.width, 4)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGRA2BGR)
        elif encoding in ['rgba8']:
            frame = frame.reshape(image_msg.height, image_msg.width, 4)
            frame = cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR)
        else:
            frame = frame.reshape(image_msg.height, image_msg.width, -1)
            if frame.shape[2] == 4:
                frame = cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR)
            elif 'rgb' in encoding:
                frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

        return frame

    except Exception as e:
        return None
    
# =========================================================
# 3. POST-PROCESSING & VẼ BOUNDING BOX
# =========================================================
def draw_detections(img, outputs, conf_threshold=0.35):
    """Gia ma Tensor YOLOv8 khong can dung OpenCV NMS."""
    annotated_img = img.copy()
    h_orig, w_orig = img.shape[:2]
    
    output = outputs[0] if isinstance(outputs, list) else outputs
    output = np.squeeze(output)

    # Xoay ma tran neu shape la [84, 8400] hoac [5, 8400]
    if output.ndim == 2 and output.shape[0] < output.shape[1]:
        output = output.T

    x_factor = w_orig / 640.0
    y_factor = h_orig / 640.0

    for row in output:
        # Score cua class bien so
        score = row[4] if len(row) == 5 else np.max(row[4:])
        
        if score >= conf_threshold:
            cx, cy, w, h = row[0], row[1], row[2], row[3]
            
            # Neu mo hinh tra ve [x1, y1, x2, y2] thay vi [cx, cy, w, h]
            if cx < w and cy < h:  
                left = int((cx - 0.5 * w) * x_factor)
                top = int((cy - 0.5 * h) * y_factor)
                width = int(w * x_factor)
                height = int(h * y_factor)
            else:
                left, top = int(cx * x_factor), int(cy * y_factor)
                width, height = int((w - cx) * x_factor), int((h - cy) * y_factor)

            cv2.rectangle(annotated_img, (left, top), (left + width, top + height), (0, 0, 255), 3)
            label = f"Bien so: {score:.2f}"
            cv2.putText(annotated_img, label, (left, max(top - 10, 15)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    return annotated_img

def frame_to_bytes(frame):
    """Chuyen anh sang Byte JPEG co kiem tra an toan."""
    if frame is None or frame.size == 0:
        return None
    success, encoded_img = cv2.imencode('.jpg', frame, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
    return encoded_img.tobytes() if success else None

In [21]:
# Khởi tạo ONNX Model Runner
ONNX_PATH = "../models/best_v8_opset15.onnx"
runner = ONNXModelRunner(ONNX_PATH)

# Khởi tạo ROS Node (tránh duplicate node)
try:
    rospy.init_node('notebook_onnx_node', anonymous=True, disable_signals=True)
except:
    pass

latest_frame = None

def camera_callback(msg):
    global latest_frame
    frame = decode_image(msg)
    if frame is not None:
        latest_frame = frame

# Đăng ký Subscriber
sub = rospy.Subscriber('/csi_cam_0/image_raw', Image, camera_callback, queue_size=1)
print("✅ Đã kết nối topic /csi_cam_0/image_raw thành công!")

[+] Đang nạp mô hình ONNX từ: ../models/best_v8_opset15.onnx
[✓] Nạp ONNX thành công!
 -> Input Name: images
 -> Input Shape mong đợi: [1, 3, 480, 640]
✅ Đã kết nối topic /csi_cam_0/image_raw thành công!


In [15]:
import rospy
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display
from sensor_msgs.msg import Image

# 1. Khởi tạo Model ONNX Runner
ONNX_PATH = "../models/best_v8_opset15.onnx"
runner = ONNXModelRunner(ONNX_PATH)

# 2. Khởi tạo ROS Node
try:
    rospy.init_node('notebook_onnx_node', anonymous=True, disable_signals=True)
except:
    pass

latest_frame = None

def camera_callback(msg):
    global latest_frame
    try:
        img_np = np.frombuffer(msg.data, dtype=np.uint8).reshape(msg.height, msg.width, -1)
        if msg.encoding == 'rgb8':
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        latest_frame = img_np
    except Exception as e:
        pass

sub = rospy.Subscriber('/csi_cam_0/image_raw', Image, camera_callback, queue_size=1)

# =========================================================
# 3. HÀM POST-PROCESSING & VẼ BOUNDING BOX (YOLOv8)
# =========================================================
def draw_detections(img, outputs, conf_threshold=0.35):
    """
    Vẽ Bounding Box cho YOLOv8 đã tích hợp sẵn End-to-End NMS.
    Output Tensor dạng: [1, N, 6] -> [x1, y1, x2, y2, confidence, class_id]
    """
    annotated_img = img.copy()
    h_orig, w_orig = img.shape[:2]
    
    # Lấy output đầu tiên và bỏ dimension batch
    output = outputs[0] if isinstance(outputs, list) else outputs
    output = np.squeeze(output)  # Dạng [N, 6]

    # Tỷ lệ scale từ 640x640 về kích thước ảnh thực tế
    x_factor = w_orig / 640.0
    y_factor = h_orig / 640.0

    for detection in output:
        # Nếu output là [x1, y1, x2, y2, score, class_id]
        score = float(detection[4])
        
        if score >= conf_threshold:
            # Lấy tọa độ x1, y1, x2, y2 trực tiếp
            x1 = int(detection[0] * x_factor)
            y1 = int(detection[1] * y_factor)
            x2 = int(detection[2] * x_factor)
            y2 = int(detection[3] * y_factor)

            # Vẽ khung hình chữ nhật
            cv2.rectangle(annotated_img, (x1, y1), (x2, y2), (0, 0, 255), 3)
            
            # In nhãn và score
            label = f"Bien so: {score:.2f}"
            cv2.putText(annotated_img, label, (x1, max(y1 - 10, 15)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    return annotated_img

[+] Đang nạp mô hình ONNX từ: ../models/best_v8_opset15.onnx
[✓] Nạp ONNX thành công!
 -> Input Name: images
 -> Input Shape mong đợi: [1, 3, 480, 640]


In [ ]:
# Tạo ảnh placeholder mặc định
blank_frame = np.zeros((480, 640, 3), dtype=np.uint8)
cv2.putText(blank_frame, "DANG CHO TOPIC CAMERA...", (120, 240), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
initial_bytes = frame_to_bytes(blank_frame)

is_running = True

# Khoi tao 2 Image Widgets voi anh mac dinh
raw_widget = widgets.Image(value=initial_bytes, format='jpeg', width=500)
processed_widget = widgets.Image(value=initial_bytes, format='jpeg', width=500)

images_box = widgets.HBox([
    widgets.VBox([widgets.HTML("<b>📷 RAW CAMERA IMAGE</b>"), raw_widget]),
    widgets.VBox([widgets.HTML("<b>🎯 MODEL DETECTION OUTPUT</b>"), processed_widget])
])

stop_btn = widgets.Button(description='🛑 Dừng Inference', button_style='danger')
fps_label = widgets.HTML(value="<b>🚀 Tốc độ xử lý: <font color='orange'>Đang kết nối...</font></b>")

def on_stop_click(b):
    global is_running
    is_running = False
    stop_btn.disabled = True
    stop_btn.description = '⏹️ Đã dừng!'

stop_btn.on_click(on_stop_click)

display(stop_btn, fps_label, images_box)

prev_time = time.time()

# Vong lap chinh
try:
    while is_running and not rospy.is_shutdown():
        if latest_frame is not None:
            current_raw = latest_frame.copy()

            # 1. Inference
            input_tensor = runner.preprocess(current_raw)
            outputs = runner.infer(input_tensor)
            
            # 2. Draw Bounding Box
            result_img = draw_detections(current_raw, outputs, conf_threshold=0.35)
            
            # 3. Tinh FPS
            curr_time = time.time()
            fps = 1.0 / (curr_time - prev_time + 1e-6)
            prev_time = curr_time
            
            cv2.putText(result_img, f"FPS: {fps:.1f}", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

            # 4. Render Bytes
            r_bytes = frame_to_bytes(current_raw)
            p_bytes = frame_to_bytes(result_img)

            if r_bytes and p_bytes:
                raw_widget.value = r_bytes
                processed_widget.value = p_bytes
                fps_label.value = f"<b>🚀 Tốc độ xử lý: <font color='lime'>{fps:.1f} FPS</font></b>"

        rospy.sleep(0.05)

except KeyboardInterrupt:
    pass

print("🟢 Đã dừng luồng hiển thị!")

Button(button_style='danger', description='🛑 Dừng Inference', style=ButtonStyle())

HTML(value="<b>🚀 Tốc độ xử lý: <font color='orange'>Đang kết nối...</font></b>")